# SHARP CEA Cartesian Extrapolation

This notebook downloads one SHARP CEA vector magnetogram, trains a Cartesian NF2 extrapolation, exports the result, and produces a few quick-look diagnostics. It mirrors the YAML example while keeping the full workflow editable in Python.

## 1. Configure The Example

Set the JSOC request, local run paths, export resolution, and height range. `RUN_DOWNLOAD` and `RUN_TRAINING` can be toggled when reusing downloaded data or an existing model state.

In [ ]:
from pathlib import Path

RUN_DOWNLOAD = True
RUN_TRAINING = True

JSOC_EMAIL = "you@example.org"
SHARP_NUM = 377
NOAA_NUM = None
T_START = "2011-02-15T00:00:00"
T_END = "2011-02-15T00:12:00"
CADENCE = "720s"
SERIES = "sharp_cea_720s"
SEGMENTS = "Br,Bp,Bt,Br_err,Bp_err,Bt_err"

RUN_DIR = Path("runs/sharp_cea")
DATA_DIR = RUN_DIR / "data"
WORK_DIR = RUN_DIR / "work"
NF2_PATH = RUN_DIR / "extrapolation_result.nf2"
EXPORT_DIR = RUN_DIR / "exports"

BR_FITS = DATA_DIR / "Br.fits"
BT_FITS = DATA_DIR / "Bt.fits"
BP_FITS = DATA_DIR / "Bp.fits"
BR_ERR_FITS = DATA_DIR / "Br_err.fits"
BT_ERR_FITS = DATA_DIR / "Bt_err.fits"
BP_ERR_FITS = DATA_DIR / "Bp_err.fits"

EXPORT_MM_PER_PIXEL = 0.72
HEIGHT_MIN = 0
HEIGHT_MAX = 80

## 2. Import Packages

W&B console capture is disabled in notebooks to keep Rich/Jupyter output stable while preserving online logging.

In [ ]:
import os
os.environ.setdefault("WANDB_CONSOLE", "off")

from pathlib import Path
from dateutil.parser import parse

from astropy import units as u
from matplotlib.colors import LogNorm
import matplotlib.pyplot as plt
import numpy as np
import wandb

import nf2
from nf2.evaluation.metric import normalized_divergence, sigma_J, theta_J, vector_norm
from nf2.evaluation.output_metrics import current_density

## 3. Download The Magnetogram

The helper downloads the SHARP CEA vector components and uncertainty maps, then selects the first complete time step for the example extrapolation.

In [ ]:
DATA_DIR.mkdir(parents=True, exist_ok=True)
if RUN_DOWNLOAD:
    nf2.download_sharp_series(
        str(DATA_DIR), JSOC_EMAIL, parse(T_START), parse(T_END),
        noaa_num=NOAA_NUM, sharp_num=SHARP_NUM, cadence=CADENCE,
        segments=SEGMENTS, series=SERIES,
    )

def find_segment(segment, fallback):
    matches = sorted(DATA_DIR.glob(f"*.{segment}.fits"))
    return matches[0] if matches else fallback

if not BR_FITS.exists():
    BR_FITS = find_segment("Br", BR_FITS)
if not BT_FITS.exists():
    BT_FITS = find_segment("Bt", BT_FITS)
if not BP_FITS.exists():
    BP_FITS = find_segment("Bp", BP_FITS)
if not BR_ERR_FITS.exists():
    BR_ERR_FITS = find_segment("Br_err", BR_ERR_FITS)
if not BT_ERR_FITS.exists():
    BT_ERR_FITS = find_segment("Bt_err", BT_ERR_FITS)
if not BP_ERR_FITS.exists():
    BP_ERR_FITS = find_segment("Bp_err", BP_ERR_FITS)

print("Field files:")
print(BR_FITS, BT_FITS, BP_FITS, sep="\n")
print("Error files:")
print(BR_ERR_FITS, BT_ERR_FITS, BP_ERR_FITS, sep="\n")

## 4. Build And Train The Configuration

The configuration includes the observed lower boundary, an explicit potential top/side boundary, and matching losses for both. Boundary and slice callbacks log validation images to W&B.

In [ ]:
losses = [
    {"type": "boundary", "name": "boundary", "weight": 1.0, "datasets": "boundary"},
    {"type": "boundary", "name": "potential_boundary", "weight": 10.0, "datasets": "potential"},
    {"type": "force_free", "name": "force_free", "weight": 1.0e-3, "datasets": ["random"]},
    {"type": "potential", "name": "potential", "weight": {"type": "step", "steps": 5000, "start": 1.0e-4, "end": 0.0}, "datasets": ["random"]},
]

config = {
    "path": str(RUN_DIR),
    "work_path": str(WORK_DIR),
    "logging": {"project": "nf2", "name": "SHARP CEA"},
    "data": {
        "geometry": "cartesian",
        "boundaries": [{
            "id": "boundary",
            "type": "sharp",
            "files": {"Br": str(BR_FITS), "Bt": str(BT_FITS), "Bp": str(BP_FITS)},
            "errors": {"Br_err": str(BR_ERR_FITS), "Bt_err": str(BT_ERR_FITS), "Bp_err": str(BP_ERR_FITS)},
        }],
        "potential_boundary": {"type": "potential", "strides": 4},
        "validation": [
            {"id": "boundary", "type": "sharp", "files": {"Br": str(BR_FITS), "Bt": str(BT_FITS), "Bp": str(BP_FITS)}, "errors": {"Br_err": str(BR_ERR_FITS), "Bt_err": str(BT_ERR_FITS), "Bp_err": str(BP_ERR_FITS)}},
            {"id": "cube", "type": "cube"},
            {"id": "slices", "type": "slices"},
        ],
    },
    "losses": losses,
    "loss_scaling": [{"type": "b_height", "name": "b_height", "loss_ids": ["force_free", "potential"]}],
    "callbacks": [{"type": "boundary", "dataset": "boundary"}, {"type": "metrics", "dataset": "cube"}, {"type": "slices", "dataset": "slices"}],
}

if RUN_TRAINING:
    nf2.run(**config)
    wandb.finish()
else:
    config

## 5. Export And Evaluate

Export the trained state to HDF5, reload one `.nf2` result, and compute standard force-free and divergence diagnostics.

In [ ]:
EXPORT_DIR.mkdir(parents=True, exist_ok=True)
nf2.export_file(str(NF2_PATH), str(EXPORT_DIR / "field.vtk"), fmt="vtk", Mm_per_pixel=EXPORT_MM_PER_PIXEL, height_range=[HEIGHT_MIN, HEIGHT_MAX], metrics=["j", "alpha", "free_energy_fft"])
nf2.export_file(str(NF2_PATH), str(EXPORT_DIR / "field.npz"), fmt="npz", Mm_per_pixel=EXPORT_MM_PER_PIXEL, height_range=[HEIGHT_MIN, HEIGHT_MAX], metrics=["j", "alpha", "free_energy_fft"])

## 6. Load The Output Cube

Load the magnetic field cube and derived metrics used by the summary table and plots.

In [ ]:
out = nf2.load(NF2_PATH)
cube = out.load_cube(
    Mm_per_pixel=EXPORT_MM_PER_PIXEL,
    height_range=[HEIGHT_MIN, HEIGHT_MAX],
    metrics=["j", "j_vec", "free_energy_fft"],
    progress=True,
)

b = cube["b"].to_value(u.G)
j = cube["metrics"]["j"].to_value(u.G / u.s)
j_vec = cube["metrics"]["j_vec"].to_value(u.G / u.s)
free_energy = cube["metrics"]["free_energy_fft"].to_value(u.erg / u.cm**3)
voxel_volume_cm3 = (EXPORT_MM_PER_PIXEL * u.Mm).to_value(u.cm) ** 3
metrics = {
    "mean_normalized_divergence": float(np.nanmean(normalized_divergence(b))),
    "theta_J_deg": float(theta_J(b, j_vec)),
    "sigma_J_1e2": float(sigma_J(b, j_vec) * 1e2),
    "mean_B_norm": float(np.nanmean(vector_norm(b))),
    "mean_J_norm": float(np.nanmean(j)),
    "total_free_energy_erg": float(np.nansum(free_energy) * voxel_volume_cm3),
}
metrics

## 7. Quick-Look Plots

The integrated current, free energy, and model boundary maps are intended as a fast visual sanity check after training.

In [ ]:
fig, axs = plt.subplots(1, 3, figsize=(15, 4))

current_map = np.nansum(j, axis=2) * EXPORT_MM_PER_PIXEL
free_energy_map = np.nansum(free_energy, axis=2) * (EXPORT_MM_PER_PIXEL * u.Mm).to_value(u.cm)
boundary_bz = b[:, :, 0, 2]

extent = [0, b.shape[0] * EXPORT_MM_PER_PIXEL, 0, b.shape[1] * EXPORT_MM_PER_PIXEL]

def log_norm(image, lower=1, upper=99):
    positive = image[np.isfinite(image) & (image > 0)]
    if positive.size == 0:
        return LogNorm(vmin=np.nextafter(0, 1), vmax=1.0)
    vmin, vmax = np.nanpercentile(positive, [lower, upper])
    return LogNorm(vmin=max(vmin, np.nextafter(0, 1)), vmax=max(vmax, vmin * 1.01))

im = axs[0].imshow(current_map.T, origin="lower", cmap="inferno", norm=log_norm(current_map), extent=extent)
axs[0].set_title("Height-integrated |J| (log)")
plt.colorbar(im, ax=axs[0], fraction=0.046, label="Height-integrated |J| [G Mm s$^{-1}$]")

im = axs[1].imshow(free_energy_map.T, origin="lower", cmap="jet", norm=log_norm(free_energy_map), extent=extent)
axs[1].set_title("Height-integrated free energy (log)")
plt.colorbar(im, ax=axs[1], fraction=0.046, label="Height-integrated free energy [erg cm$^{-2}$]")

lim = np.nanpercentile(np.abs(boundary_bz), 99)
im = axs[2].imshow(boundary_bz.T, origin="lower", cmap="RdBu_r", vmin=-lim, vmax=lim, extent=extent)
axs[2].set_title("Model boundary Bz")
plt.colorbar(im, ax=axs[2], fraction=0.046, label="$B_z$ [G]")

for ax in axs:
    ax.set_xlabel("x [Mm]")
    ax.set_ylabel("y [Mm]")
fig.tight_layout()
plt.show()